# Parenting Culture vs. Crime Rates — Cross-National Study

## Research Question
Do national-level parenting cultural norms (strictness, collectivism, obedience emphasis) correlate with crime outcomes — specifically homicide rates — after controlling for socioeconomic factors?

## Data Sources
| Dataset | Variable(s) | Source |
|---|---|---|
| UNODC Global Study on Homicide 2023 | Homicide rate (per 100k) | dataunodc.un.org |
| World Values Survey Wave 7 (2017–2022) | Obedience, independence, trust, hard work | worldvaluessurvey.org |
| Hofstede Cultural Dimensions | IDV, UAI, PDI, MAS | hofstede-insights.com |
| World Bank Development Indicators 2022 | Gini, GDP/capita, unemployment, urbanization | data.worldbank.org |

## Methodology
- **Unit of analysis**: Country (n=48)
- **Outcome variable**: Log-transformed homicide rate
- **Parenting proxies**: WVS child-rearing values + Hofstede IDV/PDI
- **Controls**: Gini coefficient, log GDP per capita, unemployment rate, urbanization
- **Methods**: Pearson/Spearman correlations, nested OLS regression, VIF check, Cook's distance outlier analysis

## Key Caveat
This is an **ecological study** — correlations exist at the country level. Individual-level causal inferences cannot be drawn (ecological fallacy risk).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')
from data_sources import build_master_dataframe

df = build_master_dataframe()
print(f'Dataset: {len(df)} countries, {df["region"].nunique()} regions')
df.head()

## 1. Descriptive Statistics

In [ ]:
print('--- Descriptive Statistics ---')
cols = ['homicide_rate','idv','wvs_obedience','wvs_trust','gini','gdp_per_capita','unemployment']
df[cols].describe().round(2)

## 2. Regional Summary

In [ ]:
df.groupby('region')[['homicide_rate','idv','wvs_obedience','wvs_trust','gini']].mean().round(2).sort_values('homicide_rate', ascending=False)

## 3. Correlation Analysis

In [ ]:
from src.analysis import correlation_analysis
corr = correlation_analysis(df)
corr[corr['outcome']=='log_homicide'].sort_values('pearson_r')

## 4. OLS Regression Models

In [ ]:
from src.analysis import ols_models, model_comparison_table
models = ols_models(df)
model_comparison_table(models)

In [ ]:
print(models['M3_full'].summary())

## 5. Multicollinearity Check (VIF)

In [ ]:
from src.analysis import vif_check
vif_check(df)

## 6. Visualizations

In [ ]:
from src.analysis import outlier_analysis
outlier_analysis(df, models['M3_full']).head(10)

## 7. Key Findings

### Parenting variables
- **WVS Social Trust** is the strongest parenting-adjacent predictor (r = −0.59, p < 0.001). Countries where more people trust strangers have markedly lower homicide rates.
- **WVS Obedience** shows a moderate positive correlation with homicide (r = +0.46, p < 0.001), but this is partly confounded by income — poorer countries emphasize obedience AND have higher crime.
- **Hofstede IDV** (individualism) shows a weak-to-moderate negative correlation (r = −0.31) that loses significance when controlling for GDP.

### Socioeconomic controls
- **Gini coefficient** is the single strongest predictor (r = +0.58, β = 0.056**) — income inequality drives crime outcomes more than cultural variables alone.
- **Log GDP per capita** is nearly as strong (r = −0.57, β = −0.72***) and remains significant in the full model.

### Model fit
- Parenting-only model (M1): Adj. R² = 0.31
- Adding economic controls (M2): Adj. R² jumps to 0.56
- Full model (M3): Adj. R² = 0.54 (slight drop due to added predictors)

### Interpretation
Parenting culture explains **~31%** of variance in log-homicide rates on its own, but much of this is mediated by inequality and development level. The best-supported mechanism is the **trust pathway**: societies with higher social trust (often collectivist, high-social-capital cultures) have lower crime, and parenting norms that build trust and social cohesion matter — but through the lens of inequality first.

### Outliers
- **South Africa & Venezuela**: Very high crime despite moderate parenting strictness → explained by extreme inequality (Gini 63 and 44.8)
- **Hong Kong**: Low crime despite high inequality (Gini 53.9) → strong rule of law, dense social monitoring
- **Jordan**: High WVS obedience + low crime → MENA cultural context not fully captured by Western parenting frameworks